# Prueba sencilla de inferencia del modelo final

El objetivo es demostrar que el modelo final exportado puede cargarse y producir predicciones sobre registros del dataset.

Modelo esperado: **MLP** seleccionado como modelo final por ROC-AUC de validación.

> Nota: si el modelo exportado fue guardado con otra versión de `scikit-learn`, puede aparecer una advertencia de compatibilidad. Para una ejecución estrictamente reproducible, usar la misma versión con la que se exportó el modelo.

In [1]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## 1. Configuración de rutas

Estructura esperada:

```text
Proyecto_Modelos_II/
    ExportedModels/
        exported_mlp.joblib
    FinalStage/
        03_prueba_inferencia_modelo_final.ipynb
```

Si el notebook está dentro de `FinalStage`, la raíz del proyecto será la carpeta padre.

In [2]:
CURRENT_DIR = Path.cwd()

# Si este notebook está dentro de FinalStage, la raíz del repo es la carpeta padre.
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "FinalStage" else CURRENT_DIR

EXPORTED_MODELS_DIR = PROJECT_ROOT / "ExportedModels"

MODEL_CANDIDATES = [
    EXPORTED_MODELS_DIR / "mlp_roc_selected_model.joblib",
    EXPORTED_MODELS_DIR / "exported_mlp.joblib",
    EXPORTED_MODELS_DIR / "mlp_model.joblib",
]

print("CURRENT_DIR:", CURRENT_DIR.resolve())
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("EXPORTED_MODELS_DIR:", EXPORTED_MODELS_DIR.resolve())

for path in MODEL_CANDIDATES:
    print(path.name, "->", path.exists())

CURRENT_DIR: C:\Users\HP\Documents\Materias_2026_1\Modelos_II\Proyecto_Modelos_II\FinalStage
PROJECT_ROOT: C:\Users\HP\Documents\Materias_2026_1\Modelos_II\Proyecto_Modelos_II
EXPORTED_MODELS_DIR: C:\Users\HP\Documents\Materias_2026_1\Modelos_II\Proyecto_Modelos_II\ExportedModels
mlp_roc_selected_model.joblib -> False
exported_mlp.joblib -> True
mlp_model.joblib -> False


## 2. Cargar el modelo final exportado

El modelo puede estar guardado como:

1. un `Pipeline` directamente, o  
2. un `dict`/`bundle` con claves como `model`, `threshold`, `model_name`, etc.

Esta celda soporta ambos casos.

In [3]:
def find_existing_path(paths):
    for path in paths:
        if path.exists():
            return path
    return None

MODEL_PATH = find_existing_path(MODEL_CANDIDATES)

if MODEL_PATH is None:
    raise FileNotFoundError(
        "No se encontró el modelo MLP exportado. "
        "Verifica que exista en ExportedModels/ con alguno de estos nombres: "
        + ", ".join(path.name for path in MODEL_CANDIDATES)
    )

artifact = joblib.load(MODEL_PATH)

if isinstance(artifact, dict):
    model = artifact.get("model")
    threshold = artifact.get("threshold", 0.5)
    model_name = artifact.get("model_name", "MLP final")
else:
    model = artifact
    threshold = 0.5
    model_name = "MLP final"

if model is None:
    raise ValueError("El artefacto cargado no contiene una clave 'model' válida.")

print("Modelo cargado desde:", MODEL_PATH)
print("Nombre del modelo:", model_name)
print("Threshold usado:", threshold)
print("Tipo de objeto:", type(model))

Modelo cargado desde: c:\Users\HP\Documents\Materias_2026_1\Modelos_II\Proyecto_Modelos_II\ExportedModels\exported_mlp.joblib
Nombre del modelo: MLP tanh regularized (64,32)
Threshold usado: 0.20999999999999996
Tipo de objeto: <class 'sklearn.pipeline.Pipeline'>


## 3. Cargar datos para probar predicciones

Primero se intenta cargar un archivo local. Si no existe, se usa la URL del dataset limpio del repositorio.

In [4]:
TARGET_COLUMN = "cardiovascular_disease"

DATA_CANDIDATES = [
    PROJECT_ROOT / "dataset" / "data_cleaned.csv",
    PROJECT_ROOT / "data" / "data_cleaned.csv",
    PROJECT_ROOT / "data_cleaned.csv",
]

DATA_URL = "https://raw.githubusercontent.com/SantCorrea802/Cardiovascular_Disease_Proyecto_Modelos_II/main/dataset/data_cleaned.csv"

data_path = find_existing_path(DATA_CANDIDATES)

if data_path is not None:
    data = pd.read_csv(data_path)
    print("Datos cargados desde archivo local:", data_path)
else:
    data = pd.read_csv(DATA_URL)
    print("Datos cargados desde URL:", DATA_URL)

print("Dimensiones:", data.shape)
data.head()

Datos cargados desde archivo local: c:\Users\HP\Documents\Materias_2026_1\Modelos_II\Proyecto_Modelos_II\dataset\data_cleaned.csv
Dimensiones: (68589, 13)


,age,gender,height,weight,systolic_blood_pressure,diastolic_blood_pressure,cholesterol,glucose,smoke,alcohol_intake,physical_activity,bmi,cardiovascular_disease
0,50,2,168,62.0,110,80,1,1,0,0,1,1,0
1,55,1,156,85.0,140,90,3,1,0,0,1,3,1
2,52,1,165,64.0,130,70,3,1,0,0,0,1,1
3,48,2,169,82.0,150,100,1,1,0,0,1,2,1
4,48,1,156,56.0,100,60,1,1,0,0,0,1,0


## 4. Preparar una muestra pequeña

Tomamos pocos registros para verificar que el modelo genera:

- probabilidad o score de riesgo
- clase predicha
- clase real
- comparación acierto/error

Esto es una prueba funcional, no una evaluación estadística.

In [5]:
N_SAMPLES = 10
RANDOM_STATE = 42

if TARGET_COLUMN not in data.columns:
    raise ValueError(f"No existe la columna objetivo: {TARGET_COLUMN}")

sample = data.sample(n=min(N_SAMPLES, len(data)), random_state=RANDOM_STATE).copy()

X_sample = sample.drop(columns=[TARGET_COLUMN])
y_sample = sample[TARGET_COLUMN].astype(int)

print("Muestra seleccionada:", X_sample.shape)
X_sample.head()

Muestra seleccionada: (10, 12)


,age,gender,height,weight,systolic_blood_pressure,diastolic_blood_pressure,cholesterol,glucose,smoke,alcohol_intake,physical_activity,bmi
16089,42,1,165,54.0,140,90,1,1,0,0,1,1
23811,56,2,182,105.0,130,100,1,1,0,0,1,3
7390,50,2,172,87.0,100,80,1,1,0,0,1,2
54931,52,1,145,45.0,110,70,1,1,0,0,0,1
29239,52,2,170,62.0,130,90,1,1,0,0,1,1


## 5. Realizar predicciones

La función intenta usar `predict_proba`. Si el modelo no la tiene, usa `decision_function` y transforma el score a una escala 0-1 con una función logística.

In [6]:
def get_decision_scores(model, X):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return np.asarray(proba).ravel()

    if hasattr(model, "decision_function"):
        raw_scores = model.decision_function(X)
        raw_scores = np.asarray(raw_scores).ravel()
        return 1.0 / (1.0 + np.exp(-raw_scores))

    # Último recurso: usar predict como score binario.
    return np.asarray(model.predict(X)).ravel()


scores = get_decision_scores(model, X_sample)

# Si el threshold guardado es extraño o no existe, se usa 0.5.
try:
    threshold_value = float(threshold)
except Exception:
    threshold_value = 0.5

predictions = (scores >= threshold_value).astype(int)

results = X_sample.copy()
results["real"] = y_sample.values
results["score_riesgo"] = scores
results["prediccion"] = predictions
results["acierto"] = results["real"] == results["prediccion"]

cols_to_show = ["real", "score_riesgo", "prediccion", "acierto"]
results[cols_to_show].sort_values("score_riesgo", ascending=False)

,real,score_riesgo,prediccion,acierto
16089,1,0.788998,1,True
11619,1,0.742184,1,True
23811,1,0.595065,1,True
56766,1,0.524431,1,True
29239,0,0.514460,1,False
15745,0,0.394145,1,False
3142,0,0.347224,1,False
7390,1,0.247315,1,True
40527,0,0.203908,0,True
54931,0,0.130166,0,True


## 6. Resumen de la prueba

Esta salida confirma que el modelo:

1. se cargó correctamente
2. aceptó registros con la estructura esperada
3. produjo scores
4. produjo clases predichas usando el threshold definido.

In [7]:
n_correct = int(results["acierto"].sum())
n_total = len(results)

print(f"Predicciones generadas: {n_total}")
print(f"Aciertos en esta muestra pequeña: {n_correct}/{n_total}")
print(f"Threshold aplicado: {threshold_value}")

summary = results[["real", "score_riesgo", "prediccion", "acierto"]].copy()
summary

Predicciones generadas: 10
Aciertos en esta muestra pequeña: 7/10
Threshold aplicado: 0.20999999999999996


,real,score_riesgo,prediccion,acierto
16089,1,0.788998,1,True
23811,1,0.595065,1,True
7390,1,0.247315,1,True
54931,0,0.130166,0,True
29239,0,0.514460,1,False
3142,0,0.347224,1,False
15745,0,0.394145,1,False
40527,0,0.203908,0,True
11619,1,0.742184,1,True
56766,1,0.524431,1,True


## 7. Prueba con un registro individual

Esta celda muestra cómo se usaría el modelo para predecir un único paciente ya representado con las mismas columnas del dataset.

In [8]:
single_patient = X_sample.iloc[[0]].copy()

single_score = get_decision_scores(model, single_patient)[0]
single_prediction = int(single_score >= threshold_value)

print("Score de riesgo:", round(float(single_score), 4))
print("Predicción:", single_prediction)
print("Interpretación:", "Con enfermedad cardiovascular" if single_prediction == 1 else "Sin enfermedad cardiovascular")

Score de riesgo: 0.789
Predicción: 1
Interpretación: Con enfermedad cardiovascular
